# 05 — BVAR Main Results

## What this notebook does
1. **Load production inputs** — use the transformed panel from notebook 03 and the frozen COVID-volatility parameters from notebook 04
2. **Define the baseline BVAR** — Minnesota prior, Jeffreys-type covariance prior, two lags, and the frozen truncated-volatility path
3. **Identify structural shocks** — recursive Cholesky ordering `[GPR, Brent, Gas, HICP, IPI, CLIFS]`
4. **Estimate the 16 baseline systems** — four recipient economies × four GPR origins
5. **Summarise the main IRFs** — posterior medians and 68% credible intervals
6. **Assess geographical heterogeneity** — exact permutation test based on detectable responses at impact and one year
7. **Export baseline results** — save IRFs, synthesis tables, heterogeneity results, figures, and the model object used by notebook 06

This notebook contains the **baseline empirical results only**. Robustness exercises are deferred to notebook 06.


## A. Setup and production inputs


In [1]:
# ============================================================
# A. SETUP AND PRODUCTION INPUTS
# ============================================================
from pathlib import Path
from itertools import product
import pickle
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Support both Google Colab and local execution
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/TFM_GitHub")
except ImportError:
    PROJECT_ROOT = Path.cwd().resolve()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "notebooks").exists():
    raise FileNotFoundError(
        f"Could not find the project under: {PROJECT_ROOT}"
    )

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
COVID_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "04_covid_volatility"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "05_bvar_main_results"
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BVAR_READY_PATH = PROCESSED_DIR / "03_bvar_ready_panel.xlsx"
COVID_THETA_PATH = COVID_OUTPUT_DIR / "PRODUCTION_covid_theta_mle.csv"

for path in [BVAR_READY_PATH, COVID_THETA_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required production input not found: {path}")

panel = pd.read_excel(BVAR_READY_PATH)
panel["quarter"] = pd.PeriodIndex(panel["quarter"].astype(str), freq="Q")
production_theta = pd.read_csv(COVID_THETA_PATH)

TARGET_COUNTRIES = ["Germany", "France", "Italy", "Spain"]
GPR_ORIGIN_COLS = {
    "Russia": "log1p_GPR_Russia",
    "China": "log1p_GPR_China",
    "MENA": "log1p_GPR_Middle_East",
    "Western": "log1p_GPR_Western_Bloc",
}
ORIGINS = list(GPR_ORIGIN_COLS.keys())
CHOLESKY_ORDER_CORE = ["dlog_Brent", "dlog_Gas", "dlog_HICP", "dlog_IPI", "CLIFS_level"]

LAG = 2
IRF_HORIZON = 20
CI_LOW_68, CI_HIGH_68 = 16, 84
HORIZONS_TO_CHECK = {"impact": 0, "1yr": 4, "2yr": 8}
REPS, BURN, SEED = 10000, 5000, 42
LAM1, LAM2, LAM3, LAM4 = 0.2, 0.5, 1.0, 100.0
ALPHA = 0

theta_lookup = {
    (row["country"], row["gpr_origin"]): (
        row["s0_2020Q1"], row["s1_2020Q2"], row["s2_2020Q3"], row["rho"]
    )
    for _, row in production_theta.iterrows()
}

expected_panel_cols = ["quarter", "country"] + list(GPR_ORIGIN_COLS.values()) + CHOLESKY_ORDER_CORE
missing = [c for c in expected_panel_cols if c not in panel.columns]
if missing:
    raise KeyError(f"Required columns missing from notebook 03 output: {missing}")
if len(theta_lookup) != 16:
    raise RuntimeError(f"Expected 16 frozen COVID parameter sets, found {len(theta_lookup)}")

print(f"Panel: {panel.shape[0]} rows x {panel.shape[1]} columns")
print(f"Sample: {panel['quarter'].min()} to {panel['quarter'].max()}")
print(f"Frozen COVID parameter sets: {len(theta_lookup)}")
print(f"BVAR: p={LAG}, reps={REPS:,}, burn-in={BURN:,}, retained={REPS-BURN:,}")
print(f"Minnesota lambdas: {LAM1}, {LAM2}, {LAM3}, {LAM4}")
print("Covariance prior: S=0, alpha=0 (Jeffreys-type)")
print("Ordering: [GPR, Brent, Gas, HICP, IPI, CLIFS]")


Mounted at /content/drive
Panel: 460 rows x 11 columns
Sample: 1997Q2 to 2025Q4
Frozen COVID parameter sets: 16
BVAR: p=2, reps=10,000, burn-in=5,000, retained=5,000
Minnesota lambdas: 0.2, 0.5, 1.0, 100.0
Covariance prior: S=0, alpha=0 (Jeffreys-type)
Ordering: [GPR, Brent, Gas, HICP, IPI, CLIFS]


## B. Baseline BVAR specification

The baseline model is a six-variable BVAR with two lags. GPR enters as `log(1+GPR)`, Brent, Gas, HICP and IPI enter in quarterly log differences, and CLIFS enters in levels. The Minnesota prior shrinks the stationary variables towards zero, while the first own lag of GPR is centred on an auxiliary AR(1) estimate. The covariance matrix uses the baseline `S=0`, `alpha=0` Jeffreys-type treatment. The COVID volatility path estimated in notebook 04 is frozen and enters the likelihood through WLS scaling.


In [2]:
# ============================================================
# B1. OLS VAR HELPER, AR(1) ESTIMATOR, COVID SCALE PATH
# ============================================================
# estimate_var_ols mirrors the function used in NB04, kept identical
# so results are directly comparable across notebooks.

def estimate_var_ols(Y, p):
    """OLS VAR(p) estimation. X ordered [const, lag1(all vars), lag2(all vars), ...]."""
    T, n = Y.shape
    X = np.column_stack([np.ones(T - p)] + [Y[p - i - 1: T - i - 1] for i in range(p)])
    Ydep = Y[p:]
    Beta = np.linalg.lstsq(X, Ydep, rcond=None)[0]
    Uhat = Ydep - X @ Beta
    Sigma = (Uhat.T @ Uhat) / (T - p)
    return Beta, Uhat, Sigma, X, Ydep

def estimate_ar1(x):
    y = x[1:]; x_lag = x[:-1]
    X = np.column_stack([np.ones(len(x_lag)), x_lag])
    return np.linalg.lstsq(X, y, rcond=None)[0][1]

def build_truncated_covid_scale_path(quarters, s0, s1, s2, rho):
    """Identical to the function used in NB04 -- must not diverge."""
    periods = pd.PeriodIndex(pd.Series(quarters).astype(str), freq="Q")
    path = np.ones(len(periods), dtype=float)
    q0, q1, q2 = pd.Period("2020Q1", "Q"), pd.Period("2020Q2", "Q"), pd.Period("2020Q3", "Q")
    trunc = pd.Period("2021Q4", "Q")
    for idx, q in enumerate(periods):
        if q == q0:
            path[idx] = s0
        elif q == q1:
            path[idx] = s1
        elif q == q2:
            path[idx] = s2
        elif q > q2 and q < trunc:
            j = q.ordinal - q2.ordinal
            path[idx] = 1.0 + (s2 - 1.0) * (rho ** j)
    return path


In [3]:
# ============================================================
# B2. MINNESOTA PRIOR -- MIXED SPECIFICATION
# ============================================================
# Prior mean of the own first-lag coefficient:
#   - GPR (index 0 in the Cholesky order): AR(1) coefficient estimated
#     from the data. GPR is persistent but mean-reverting (piecewise
#     stationarity confirmed via PELT / manual Zivot-Andrews), so
#     neither a pure random walk (1) nor white noise (0) is appropriate.
#     This follows the same principle as Bondarenko et al. (2026), who
#     relax the random-walk assumption for their GPR indicator, without
#     importing their specific numerical value (calibrated for a
#     different series).
#   - The other 5 variables (already stationary by construction --
#     log-differenced or bounded index): white noise (0), standard
#     practice for pre-stationary series in Minnesota-type priors
#     (Banbura, Giannone & Reichlin, 2010, Journal of Applied
#     Econometrics, 25(1), 71-92).
# All prior variances follow the standard Minnesota decay structure.

def build_minnesota_prior_mixed(s, n, p, gpr_idx, phi_gpr,
                                 lam1=LAM1, lam2=LAM2, lam3=LAM3, lam4=LAM4):
    """
    Coefficient vector per equation ordered as in estimate_var_ols's X:
    [const, x_1..x_n at lag1, x_1..x_n at lag2, ..., x_1..x_n at lag p]
    Returns B0 (prior mean, stacked equation by equation) and H
    (diagonal prior covariance), each of length n*(1+n*p).
    """
    k = 1 + n * p
    B0 = np.zeros(n * k)
    H_diag = np.zeros(n * k)

    for i in range(n):
        base = i * k
        H_diag[base] = (s[i] * lam4) ** 2

        for l in range(1, p + 1):
            for j in range(n):
                col = base + 1 + (l - 1) * n + j
                if j == i:
                    H_diag[col] = (lam1 / (l ** lam3)) ** 2
                    if l == 1 and i == gpr_idx:
                        B0[col] = phi_gpr
                else:
                    H_diag[col] = ((s[i] * lam1 * lam2) / (s[j] * (l ** lam3))) ** 2

    return B0, np.diag(H_diag)


In [4]:
# ============================================================
# B3. GIBBS SAMPLER -- MINNESOTA BVAR WITH INDEPENDENT NORMAL
# COEFFICIENT PRIOR AND JEFFREYS-TYPE COVARIANCE PRIOR
# ============================================================
# Prior structure:
#   beta ~ N(B0, H), independent of Sigma a priori (H is fixed and does
#   not scale with Sigma -- this is NOT a natural-conjugate Normal-
#   Inverse-Wishart prior, which would tie beta's prior covariance to
#   Sigma via a Kronecker product).
#   p(Sigma) is set via S=0, alpha=0. With this baseline choice
#   (alpha=0), this reduces to p(Sigma) ~ |Sigma|^{-(N+1)/2}: the
#   improper Jeffreys-type prior for the covariance matrix, not a
#   proper Inverse-Wishart prior. Because beta's prior is
#   not jointly conjugate with this Sigma prior, the joint posterior has
#   no closed form; the CONDITIONAL posteriors do (beta | Sigma, Y ~
#   Normal; Sigma | beta, Y ~ Inverse-Wishart), which is exactly what
#   the Gibbs sampler below exploits.
# COVID-corrected via WLS.
#
# Two fixes relative to earlier versions:
#
# 1. Minnesota prior scale (s = residual std per equation) is ALWAYS
#    computed from the UNADJUSTED baseline regression (no COVID scaling),
#    never from the WLS-transformed system. This keeps the prior
#    identical regardless of the COVID treatment, isolating the effect
#    of the volatility correction to the likelihood only -- same fix
#    already applied in notebook 04's production sampler.
#
# 2. The covariance-prior degrees-of-freedom adjustment is fixed at alpha=0.

def IWPQ(v, ixpx, rng):
    """Draw from Inverse-Wishart(v, ixpx^{-1}).

    Constructs a Wishart(v, ixpx) draw as the sum of v outer products
    of independent Gaussian vectors with covariance ixpx, generated
    using the Cholesky factor of ixpx. The resulting Wishart draw is
    then inverted to obtain an Inverse-Wishart(v, ixpx^{-1}) draw.

    This is the direct Gaussian construction of a Wishart draw, not a
    Bartlett decomposition.
    """
    k = ixpx.shape[0]
    L = np.linalg.cholesky(ixpx)
    A = np.zeros((v, k))
    for i in range(v):
        A[i] = L @ rng.standard_normal(k)
    return np.linalg.inv(A.T @ A)

def bvar_minnesota_gibbs_covid(Y, quarters, p, gpr_idx, theta,
                                reps=REPS, burn=BURN,
                                lam1=LAM1, lam2=LAM2, lam3=LAM3, lam4=LAM4,
                                seed=SEED):
    """
    COVID-corrected BVAR: the system is WLS-transformed by the frozen
    scale path s_t (from notebook 04) before estimation. The Minnesota prior
    scale is computed separately, from the unadjusted baseline
    regression, so it does not change across COVID treatments.

    Sampler: `reps` total Gibbs iterations (default 10,000), the first
    `burn` discarded as burn-in (default 5,000), leaving `reps - burn`
    retained posterior draws (default 5,000). No thinning is applied.
    """
    rng = np.random.default_rng(seed)
    T, n = Y.shape

    # Prior scale: ALWAYS from the unadjusted baseline regression.
    _, _, Sigma_baseline, _, _ = estimate_var_ols(Y, p)
    s_prior = np.sqrt(np.diag(Sigma_baseline))
    phi_gpr = estimate_ar1(Y[:, gpr_idx])

    # WLS transform using the frozen COVID scale path.
    s0, s1, s2, rho = theta
    full_scale = build_truncated_covid_scale_path(quarters, s0, s1, s2, rho)
    X_raw = np.column_stack([np.ones(T - p)] + [Y[p - i - 1: T - i - 1] for i in range(p)])
    Ydep_raw = Y[p:]
    s_eff = full_scale[p:]
    X = X_raw / s_eff[:, None]
    Ydep = Ydep_raw / s_eff[:, None]
    Beta_ols = np.linalg.lstsq(X, Ydep, rcond=None)[0]

    B0, H = build_minnesota_prior_mixed(s_prior, n, p, gpr_idx, phi_gpr, lam1, lam2, lam3, lam4)
    Hinv = np.linalg.inv(H)

    alpha = ALPHA
    S = np.zeros((n, n))

    betaols_vec = Beta_ols.T.reshape(-1)
    Sigma = (Ydep - X @ Beta_ols).T @ (Ydep - X @ Beta_ols) / (T - p)
    k = 1 + n * p
    XtX = X.T @ X

    beta_draws = np.zeros((reps - burn, n, k))
    sigma_draws = np.zeros((reps - burn, n, n))

    for it in range(reps):
        SigInv = np.linalg.inv(Sigma)
        V = np.linalg.inv(Hinv + np.kron(SigInv, XtX))
        M = V @ (Hinv @ B0 + np.kron(SigInv, XtX) @ betaols_vec)
        beta_vec = M + np.linalg.cholesky(V) @ rng.standard_normal(n * k)
        Beta = beta_vec.reshape(n, k).T

        Uhat_it = Ydep - X @ Beta
        scale = Uhat_it.T @ Uhat_it + S
        Sigma = IWPQ(T - p + alpha, np.linalg.inv(scale), rng)

        if it >= burn:
            beta_draws[it - burn] = Beta.T
            sigma_draws[it - burn] = Sigma

    return beta_draws, sigma_draws, phi_gpr


## C. Structural identification and impulse responses

Structural shocks are identified recursively with GPR ordered first, followed by Brent, Gas, HICP, IPI and CLIFS. For each retained posterior draw, the Cholesky factor of the covariance matrix is used to construct structural IRFs through horizon 20. Baseline inference uses posterior medians and 68% credible intervals.


In [5]:
# ============================================================
# B4. STRUCTURAL IRFs FROM POSTERIOR DRAWS (CHOLESKY, GPR FIRST)
# ============================================================
def compute_structural_irfs(Phis, Sigma, horizon, n, p):
    """Wold recursion + Cholesky structural IRFs (same as NB04)."""
    Psi = [np.eye(n)]
    for j in range(1, horizon + 1):
        psi_j = np.zeros((n, n))
        for i in range(1, min(j, p) + 1):
            psi_j += Phis[i - 1] @ Psi[j - i]
        Psi.append(psi_j)
    Q = np.linalg.cholesky(Sigma)
    return np.array([psi @ Q for psi in Psi])

def phis_from_beta(Beta_T, n, p):
    return [Beta_T[:, 1 + i*n : 1 + (i+1)*n] for i in range(p)]

def bvar_structural_irfs(beta_draws, sigma_draws, n, p, horizon):
    n_draws = beta_draws.shape[0]
    Theta_draws = np.zeros((n_draws, horizon + 1, n, n))
    for d in range(n_draws):
        Phis = phis_from_beta(beta_draws[d], n, p)
        Theta_draws[d] = compute_structural_irfs(Phis, sigma_draws[d], horizon, n, p)
    return Theta_draws

IRF_HORIZON = 20
CI_LOW_68, CI_HIGH_68 = 16, 84  # 68% credible interval (Bondarenko et al., 2026 convention)

## D. Estimate the 16 baseline BVARs


In [6]:
# ============================================================
# D. ESTIMATE THE 16 BASELINE BVAR SYSTEMS
# ============================================================
bvar_results = {}

for country in TARGET_COUNTRIES:
    sub = panel[panel["country"] == country].sort_values("quarter").reset_index(drop=True)

    for origin, gpr_col in GPR_ORIGIN_COLS.items():
        var_order = [gpr_col] + CHOLESKY_ORDER_CORE
        data_full = sub[["quarter"] + var_order].dropna()
        quarters = data_full["quarter"].astype(str).reset_index(drop=True)
        data = data_full[var_order].to_numpy(dtype=float)
        theta = theta_lookup[(country, origin)]

        beta_draws, sigma_draws, phi_gpr = bvar_minnesota_gibbs_covid(
            data, quarters, p=LAG, gpr_idx=0, theta=theta,
            reps=REPS, burn=BURN, seed=SEED
        )
        theta_draws = bvar_structural_irfs(
            beta_draws, sigma_draws, n=len(var_order), p=LAG, horizon=IRF_HORIZON
        )
        bvar_results[(country, origin)] = (theta_draws, phi_gpr, var_order)
        print(
            f"{country:8s} - {origin:7s} | AR(1) prior mean={phi_gpr:.3f} | "
            f"{theta_draws.shape[0]:,} retained IRF draws"
        )

if len(bvar_results) != 16:
    raise RuntimeError(f"Expected 16 BVAR systems, obtained {len(bvar_results)}")

print("\nAll 16 baseline BVAR systems estimated.")


Germany  - Russia  | AR(1) prior mean=0.906 | 5,000 retained IRF draws
Germany  - China   | AR(1) prior mean=0.833 | 5,000 retained IRF draws
Germany  - MENA    | AR(1) prior mean=0.694 | 5,000 retained IRF draws
Germany  - Western | AR(1) prior mean=0.873 | 5,000 retained IRF draws
France   - Russia  | AR(1) prior mean=0.878 | 5,000 retained IRF draws
France   - China   | AR(1) prior mean=0.870 | 5,000 retained IRF draws
France   - MENA    | AR(1) prior mean=0.809 | 5,000 retained IRF draws
France   - Western | AR(1) prior mean=0.851 | 5,000 retained IRF draws
Italy    - Russia  | AR(1) prior mean=0.794 | 5,000 retained IRF draws
Italy    - China   | AR(1) prior mean=0.776 | 5,000 retained IRF draws
Italy    - MENA    | AR(1) prior mean=0.502 | 5,000 retained IRF draws
Italy    - Western | AR(1) prior mean=0.738 | 5,000 retained IRF draws
Spain    - Russia  | AR(1) prior mean=0.787 | 5,000 retained IRF draws
Spain    - China   | AR(1) prior mean=0.719 | 5,000 retained IRF draws
Spain 

In [7]:
# ============================================================
# D2. SAMPLE AND DRAW VALIDATION
# ============================================================
validation_rows = []
for country in TARGET_COUNTRIES:
    sub = panel[panel["country"] == country].sort_values("quarter").reset_index(drop=True)
    for origin, gpr_col in GPR_ORIGIN_COLS.items():
        var_order = [gpr_col] + CHOLESKY_ORDER_CORE
        n_obs = len(sub[["quarter"] + var_order].dropna())
        theta_draws, phi_gpr, _ = bvar_results[(country, origin)]
        validation_rows.append({
            "country": country, "gpr_origin": origin,
            "T": n_obs, "T_effective": n_obs - LAG,
            "retained_draws": theta_draws.shape[0], "phi_gpr": phi_gpr,
        })
validation = pd.DataFrame(validation_rows)
display(validation)
assert validation["T"].eq(115).all(), "Each baseline system should contain 115 quarterly observations."
assert validation["retained_draws"].eq(REPS - BURN).all()
print("All systems passed sample and posterior-draw checks.")


,country,gpr_origin,T,T_effective,retained_draws,phi_gpr
0,Germany,Russia,115,113,5000,0.906100
1,Germany,China,115,113,5000,0.833388
2,Germany,MENA,115,113,5000,0.693985
3,Germany,Western,115,113,5000,0.872792
4,France,Russia,115,113,5000,0.877628
5,France,China,115,113,5000,0.870367
6,France,MENA,115,113,5000,0.808851
7,France,Western,115,113,5000,0.851218
8,Italy,Russia,115,113,5000,0.793868
9,Italy,China,115,113,5000,0.775831


All systems passed sample and posterior-draw checks.


## E. Main structural IRF results

The following figures show the response to a positive GPR shock for each country-origin system. The solid line is the posterior median and the shaded region is the 68% credible interval. Figures are generated from the same posterior draws used in the synthesis tables below.


In [8]:
# ============================================================
# E. SAVE THE 16 BASELINE STRUCTURAL IRF FIGURES
# ============================================================
if FIGURE_DIR.exists():
    shutil.rmtree(FIGURE_DIR)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

for country in TARGET_COUNTRIES:
    for origin in ORIGINS:
        theta_draws, phi_gpr, var_order = bvar_results[(country, origin)]
        median = np.median(theta_draws, axis=0)
        low = np.percentile(theta_draws, CI_LOW_68, axis=0)
        high = np.percentile(theta_draws, CI_HIGH_68, axis=0)

        fig, axes = plt.subplots(2, 3, figsize=(14, 8))
        axes = axes.flatten()
        for i, var_name in enumerate(var_order):
            ax = axes[i]
            ax.plot(range(IRF_HORIZON + 1), median[:, i, 0])
            ax.fill_between(range(IRF_HORIZON + 1), low[:, i, 0], high[:, i, 0], alpha=0.25)
            ax.axhline(0, linewidth=0.8)
            ax.set_title(var_name, fontsize=10)
            ax.set_xlabel("Horizon (quarters)", fontsize=8)

        fig.suptitle(
            f"{country} - {origin}: structural IRFs, 68% credible interval",
            fontsize=13, fontweight="bold"
        )
        plt.tight_layout()
        filename = f"irf_{country.lower()}_{origin.lower()}.png"
        plt.savefig(FIGURE_DIR / filename, dpi=150, bbox_inches="tight")
        plt.close(fig)

print(f"Saved 16 baseline IRF figures to: {FIGURE_DIR}")


Saved 16 baseline IRF figures to: /content/drive/MyDrive/TFM_GitHub/outputs/05_bvar_main_results/figures


## F. Baseline synthesis

For each of the 16 systems, the five non-GPR responses are summarised at impact (`h=0`), one year (`h=4`) and two years (`h=8`). This gives 240 country-origin-response-horizon combinations. A response is labelled detectable when its 68% credible interval excludes zero.


In [9]:
# ============================================================
# F. BASELINE SYNTHESIS TABLE -- 240 COMBINATIONS
# ============================================================
rows = []
for (country, origin), (theta_draws, phi_gpr, var_order) in bvar_results.items():
    median = np.median(theta_draws, axis=0)
    low = np.percentile(theta_draws, CI_LOW_68, axis=0)
    high = np.percentile(theta_draws, CI_HIGH_68, axis=0)

    for i, var_name in enumerate(var_order[1:], start=1):
        for h_label, h in HORIZONS_TO_CHECK.items():
            med = median[h, i, 0]
            lo = low[h, i, 0]
            hi = high[h, i, 0]
            rows.append({
                "country": country,
                "gpr_origin": origin,
                "response_var": var_name,
                "horizon": h_label,
                "h": h,
                "median": med,
                "ci_low_68": lo,
                "ci_high_68": hi,
                "sign": "+" if med > 0 else "-",
                "significant_68": (lo > 0) or (hi < 0),
            })

bvar_synthesis_240 = (
    pd.DataFrame(rows)
    .sort_values(["country", "gpr_origin", "response_var", "h"])
    .reset_index(drop=True)
)
assert len(bvar_synthesis_240) == 240

SYNTHESIS_PATH = OUTPUT_DIR / "BVAR_synthesis_all_240.csv"
bvar_synthesis_240.to_csv(SYNTHESIS_PATH, index=False)
print(f"Saved {len(bvar_synthesis_240)} baseline combinations to: {SYNTHESIS_PATH}")
display(bvar_synthesis_240.head(20))


Saved 240 baseline combinations to: /content/drive/MyDrive/TFM_GitHub/outputs/05_bvar_main_results/BVAR_synthesis_all_240.csv


,country,gpr_origin,response_var,horizon,h,median,ci_low_68,ci_high_68,sign,significant_68
0,France,China,CLIFS_level,impact,0,-0.002436,-0.007550,0.002724,-,False
1,France,China,CLIFS_level,1yr,4,-0.004455,-0.008719,-0.000384,-,True
2,France,China,CLIFS_level,2yr,8,-0.003802,-0.007850,-0.000198,-,True
3,France,China,dlog_Brent,impact,0,0.003472,-0.010692,0.018438,+,False
4,France,China,dlog_Brent,1yr,4,-0.004200,-0.008818,0.000416,-,False
5,France,China,dlog_Brent,2yr,8,-0.002804,-0.006211,0.000336,-,False
6,France,China,dlog_Gas,impact,0,-0.009757,-0.026150,0.006748,-,False
7,France,China,dlog_Gas,1yr,4,-0.000635,-0.008371,0.006824,-,False
8,France,China,dlog_Gas,2yr,8,0.000289,-0.004956,0.005284,+,False
9,France,China,dlog_HICP,impact,0,0.000350,0.000049,0.000646,+,True


## G. Geographical heterogeneity

The baseline heterogeneity exercise counts detectable responses for each GPR origin at impact and one year. There are 5 response variables × 4 recipient economies × 2 horizons = 40 possible detectable responses per origin. Under the null, GPR origins are exchangeable within each recipient economy. The exact reference distribution enumerates all `4^4 = 256` possible origin assignments across the four countries.


In [10]:
# ============================================================
# G1. DETECTABLE-RESPONSE COUNTS BY COUNTRY AND ORIGIN
# ============================================================
impact_1yr = bvar_synthesis_240[bvar_synthesis_240["horizon"].isin(["impact", "1yr"])].copy()
score_matrix = (
    impact_1yr.groupby(["country", "gpr_origin"])["significant_68"]
    .sum()
    .unstack("gpr_origin")
    .reindex(index=TARGET_COUNTRIES, columns=ORIGINS)
    .astype(int)
)
observed_counts = score_matrix.sum(axis=0).astype(int)

print("Detectable responses by country and origin (impact + one year):")
display(score_matrix)
print("Total by origin:")
display(observed_counts.to_frame("detectable_count"))


Detectable responses by country and origin (impact + one year):


gpr_origin,Russia,China,MENA,Western
country,,,,
Germany,6,8,3,3
France,5,3,2,3
Italy,4,2,2,2
Spain,8,2,4,3


Total by origin:


,detectable_count
gpr_origin,
Russia,23
China,15
MENA,11
Western,11


In [11]:
# ============================================================
# G2. EXACT PERMUTATION TEST -- 4^4 = 256 ASSIGNMENTS
# ============================================================
focal_null = np.array([
    sum(score_matrix.loc[country, selected_origin]
        for country, selected_origin in zip(TARGET_COUNTRIES, selection))
    for selection in product(ORIGINS, repeat=len(TARGET_COUNTRIES))
], dtype=int)

permutation_results = pd.DataFrame({
    "gpr_origin": ORIGINS,
    "observed_count": [int(observed_counts[o]) for o in ORIGINS],
    "null_mean": [float(focal_null.mean())] * len(ORIGINS),
    "exact_p_value": [float(np.mean(focal_null >= observed_counts[o])) for o in ORIGINS],
    "numerator": [int(np.sum(focal_null >= observed_counts[o])) for o in ORIGINS],
    "denominator": [len(focal_null)] * len(ORIGINS),
})

EXPECTED_COUNTS = {"Russia": 23, "China": 15, "MENA": 11, "Western": 11}
EXPECTED_P = {"Russia": 7/256, "China": 139/256, "MENA": 232/256, "Western": 232/256}

for origin in ORIGINS:
    if int(observed_counts[origin]) != EXPECTED_COUNTS[origin]:
        raise RuntimeError(f"Baseline count for {origin} does not reproduce the final TFM result.")
    p = permutation_results.loc[permutation_results["gpr_origin"] == origin, "exact_p_value"].iloc[0]
    if not np.isclose(p, EXPECTED_P[origin], rtol=0, atol=1e-12):
        raise RuntimeError(f"Permutation p-value for {origin} does not reproduce the final TFM result.")

PERMUTATION_PATH = OUTPUT_DIR / "exact_permutation_test.csv"
permutation_results.to_csv(PERMUTATION_PATH, index=False)
display(permutation_results)
print(f"Exact reference distribution: {len(focal_null)} assignments")
print("Baseline heterogeneity results reproduce the final TFM counts and p-values.")


,gpr_origin,observed_count,null_mean,exact_p_value,numerator,denominator
0,Russia,23,15.0,0.027344,7,256
1,China,15,15.0,0.542969,139,256
2,MENA,11,15.0,0.906250,232,256
3,Western,11,15.0,0.906250,232,256


Exact reference distribution: 256 assignments
Baseline heterogeneity results reproduce the final TFM counts and p-values.


## H. Export baseline results

The complete posterior IRF object is stored for notebook 06 so robustness checks can compare alternative specifications with the same baseline without re-estimating these 16 systems.


In [12]:
# ============================================================
# H. EXPORT BASELINE RESULTS
# ============================================================
RESULTS_PATH = OUTPUT_DIR / "bvar_results_main.pkl"
SCORE_PATH = OUTPUT_DIR / "heterogeneity_score_matrix.csv"
VALIDATION_PATH = OUTPUT_DIR / "baseline_system_validation.csv"

with open(RESULTS_PATH, "wb") as f:
    pickle.dump(bvar_results, f)
score_matrix.to_csv(SCORE_PATH)
validation.to_csv(VALIDATION_PATH, index=False)

print(f"Saved baseline BVAR object: {RESULTS_PATH}")
print(f"Saved synthesis table: {SYNTHESIS_PATH}")
print(f"Saved permutation results: {PERMUTATION_PATH}")
print(f"Saved heterogeneity matrix: {SCORE_PATH}")
print(f"Saved validation table: {VALIDATION_PATH}")
print(f"Saved IRF figures: {FIGURE_DIR}")


Saved baseline BVAR object: /content/drive/MyDrive/TFM_GitHub/outputs/05_bvar_main_results/bvar_results_main.pkl
Saved synthesis table: /content/drive/MyDrive/TFM_GitHub/outputs/05_bvar_main_results/BVAR_synthesis_all_240.csv
Saved permutation results: /content/drive/MyDrive/TFM_GitHub/outputs/05_bvar_main_results/exact_permutation_test.csv
Saved heterogeneity matrix: /content/drive/MyDrive/TFM_GitHub/outputs/05_bvar_main_results/heterogeneity_score_matrix.csv
Saved validation table: /content/drive/MyDrive/TFM_GitHub/outputs/05_bvar_main_results/baseline_system_validation.csv
Saved IRF figures: /content/drive/MyDrive/TFM_GitHub/outputs/05_bvar_main_results/figures


## Baseline findings

The notebook reproduces the final baseline empirical pipeline. The interpretation of individual responses is based on the posterior medians and 68% credible intervals exported above. The geographical test is a concentration test: it asks whether detectable responses are unusually concentrated in a particular GPR origin under within-country exchangeability; it does not invalidate responses associated with the other origins.
